모델 배포 개론 08  
Last modified : 2026.03   
작성 : 박광성 (모두의연구소)  
수정 : 김지성 박기웅 (모두의연구소)  

In [ ]:
# 실행 환경 준비 — 노트북 맨 처음에 한 번 실행하세요.
import sys, subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "fastapi>=0.115,<1", "uvicorn>=0.30,<1", "pydantic>=2.8,<3",
    "transformers>=4.45,<5", "torch>=2.2", "sentencepiece>=0.2",
    "safetensors>=0.4", "streamlit>=1.38,<2", "requests>=2.32,<3",
    "httpx>=0.27,<1", "accelerate>=0.34"
], check=True)

# 서버 실행 도우미 — 노트북 맨 처음에 한 번 실행하세요.
# 노트북 안에서 uvicorn 서버를 띄우고 멈추는 함수를 정의합니다.
import os, sys, asyncio, threading, time, socket, contextlib
import uvicorn

# 작업 디렉터리를 app/ 가 있는 위치로 맞춥니다 (notebooks/ 안에서 열어도 동작).
if not os.path.isdir('app') and os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
# 코드를 저장할 폴더를 미리 만들어 둡니다.
for _d in ('app', 'models', 'data', 'frontend'):
    os.makedirs(_d, exist_ok=True)

_SERVERS = {}  # port -> (server, thread)

def _port_open(host, port):
    with contextlib.closing(socket.socket()) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def stop_server(port=8000):
    """실행 중인 서버를 멈춥니다."""
    entry = _SERVERS.pop(port, None)
    if not entry:
        return
    server, thread = entry
    server.should_exit = True
    for _ in range(50):
        if not thread.is_alive():
            break
        time.sleep(0.1)

def serve_in_thread(app, host='127.0.0.1', port=8000, log_level='warning'):
    """백그라운드에서 uvicorn 서버를 띄웁니다.

    app: FastAPI 객체 또는 'app.main:app' 같은 import 경로.
    같은 포트에 서버가 이미 있으면 먼저 멈추고 새로 띄웁니다.
    """
    stop_server(port)
    if _port_open(host, port):
        print(f'⚠️ 포트 {port}를 다른 프로세스가 사용 중입니다 (다른 노트북의 서버일 가능성).')
        print('   그 노트북에서 stop_server(8000)을 실행하거나 커널을 종료한 뒤, 이 셀을 다시 실행하세요.')
        return None
    if isinstance(app, str):
        sys.modules.pop(app.split(':')[0], None)   # 파일을 다시 저장한 경우 최신 내용 반영
    for _ in range(50):
        if not _port_open(host, port):
            break
        time.sleep(0.1)
    config = uvicorn.Config(app, host=host, port=port, log_level=log_level, loop='asyncio')
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    def _run():
        # Windows는 SelectorEventLoop, 그 외는 기본 이벤트 루프를 사용합니다.
        if sys.platform == 'win32':
            loop = asyncio.SelectorEventLoop()
        else:
            loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(server.serve())
    thread = threading.Thread(target=_run, daemon=True)
    thread.start()
    _SERVERS[port] = (server, thread)
    # 모델 로드 때문에 기동이 느릴 수 있다 — 최대 5분 대기 (첫 실행은 다운로드 포함)
    for i in range(600):
        if _port_open(host, port):
            print(f'서버 실행됨: http://{host}:{port}')
            return server
        if not thread.is_alive():
            print('서버 스레드가 종료됐습니다. 위 로그를 확인하세요.')
            return server
        if i > 0 and i % 20 == 0:
            print(f'  ... 모델 로드 중 ({i//2}초 경과)')
        time.sleep(0.5)
    print('5분 내에 서버가 시작되지 않았습니다. 위 로그를 확인하세요.')
    return server

print('서버 도우미 준비 완료 (serve_in_thread, stop_server)')

# Day 8 — 자율 프로젝트: 투자 글 요약 서비스 만들기

---

> **오늘의 목표**
>
> Day 1~7에서 배운 기술을 조합하여, 사용자가 붙여 넣은 한국어 투자 글을 요약하는 서비스를 만듭니다.  
> Hugging Face 요약 모델, FastAPI, API Key 인증, 비동기 추론, Streamlit을 하나의 서비스로 연결합니다.

---



## 1. 프로젝트 요구사항

---

### 1.1 조건

Day 5(주택 가격 예측)와 Day 6~7(이미지 분류 / 챗봇)에서 만든 서비스와 **동일한 구조**를 기본 베이스로 합니다.

```
필수 구현 항목:

1. FastAPI 백엔드
   - 추론 엔드포인트 (POST /predict)
   - Pydantic으로 입력 검증
   - 비동기 추론 (run_in_executor)

2. API Key 인증
   - Day 6의 auth.py 재사용

3. Streamlit 프론트엔드
   - 사용자 입력 → API 호출 → 결과 표시

4. 에러 처리
   - 잘못된 입력, 모델 에러 시 적절한 HTTP 상태 코드 반환
```



### 1.2 제한 사항

```
하지 않는 것:

- 모델 학습 (사전학습 모델을 가져다 씁니다)
- Docker 패키징 (MLOps 과정에서 다룹니다)
- 데이터베이스 연동
```



### 1.3 평가 기준

```
✅ 서버가 정상적으로 실행되는가?
✅ Swagger UI에서 추론이 동작하는가?
✅ API Key 없이 요청하면 401이 반환되는가?
✅ 잘못된 입력에 대해 적절한 에러 메시지가 나오는가?
✅ Streamlit UI에서 입력 → 결과 확인이 가능한가?
```

---

## 2. 모델 선택 가이드

---

### 2.1 Hugging Face에서 모델 찾기

[Hugging Face Models](https://huggingface.co/models)에서 사전학습 모델을 선택합니다.
모델 학습은 하지 않고, `from_pretrained()`으로 바로 사용할 수 있는 모델을 고릅니다.

**모델 선택 시 확인할 것:**

```
1. 태스크가 명확한가? (text-classification, image-classification, summarization 등)
2. 한국어를 지원하는가? (필수는 아니지만, 데모가 직관적입니다)
3. 모델 크기가 적당한가? (CPU 환경이면 500MB 이하를 권장합니다)
4. pipeline()으로 바로 사용 가능한가?
```



### 2.2 도메인별 추천 예시

아래는 예시일 뿐입니다. **본인이 관심 있는 도메인을 자유롭게 선택하세요.**

| 도메인 | 태스크 | 추천 모델 (예시) |
|---|---|---|
| 감정 분석 | `text-classification` | `snunlp/KR-FinBert-SC` |
| 뉴스 분류 | `text-classification` | 원하는 분류 모델 |
| 텍스트 요약 | `summarization` | `eenzeenee/t5-base-korean` |
| 번역 | `translation` | `Helsinki-NLP` 시리즈 |
| 이미지 분류 | `image-classification` | `google/vit-base-patch16` |
| 객체 탐지 | `object-detection` | `facebook/detr-resnet-50` |
| 질의 응답 | `question-answering` | 원하는 QA 모델 |



### 2.3 모델 동작 확인

선택 모델은 `eenzeenee/t5-small-korean-summarization`입니다. 한국어 텍스트 요약 태스크에 바로 사용할 수 있고, 소형 T5 모델이라 수업용 CPU/Colab 환경에서 비교적 부담이 작습니다. 서버 코드를 작성하기 전에 아래 셀에서 투자 관련 예문으로 단독 추론을 확인합니다.


In [ ]:
from transformers import pipeline

MODEL_ID = 'eenzeenee/t5-small-korean-summarization'
summarizer = pipeline('summarization', model=MODEL_ID, tokenizer=MODEL_ID, device=-1)
sample_text = '''삼성전자는 올해 반도체 수요가 회복될 것으로 전망했다. 회사는 인공지능 서버용 고대역폭 메모리 공급 확대를 추진하고 있다. 다만 환율 변동과 글로벌 경기 둔화는 실적의 위험 요인으로 언급됐다. 이 글은 특정 종목의 매수나 매도를 권유하지 않는다.'''
result = summarizer('summarize: ' + sample_text, max_length=80, min_length=10, num_beams=4, do_sample=False, no_repeat_ngram_size=3)
print(result[0]['summary_text'])

예시 코드입니다. 샘플 이미지 파일이 있어야 실행됩니다.

```python
# 예시: 이미지 분류
from transformers import pipeline

classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

result = classifier("test_image.jpg")
print(result)
# [{'label': 'tabby cat', 'score': 0.82}, ...]
```

> **확인 기준:** 아래 셀에서 한국어 요약문이 출력되면 서버 코드 작성으로 넘어갑니다.  
> 최초 실행 시 Hugging Face 모델 다운로드 때문에 시간이 걸릴 수 있습니다.



---

## 3. 프로젝트 뼈대 코드

---



### 3.1 폴더 구조

In [ ]:
from pathlib import Path

dirs = ["app", "models", "frontend"]
for directory in dirs:
    Path(directory).mkdir(exist_ok=True)
Path("app/__init__.py").touch()

print("프로젝트 구조:")
print("""
investment-summarizer/
├── 📁 app/
│   ├── __init__.py
│   ├── auth.py
│   ├── schemas.py
│   ├── model_service.py
│   └── main.py
├── 📁 models/
└── 📁 frontend/
    └── app.py
""")


### 3.2 auth.py — 재사용

In [ ]:
%%writefile app/auth.py
import os
from fastapi import Header, HTTPException, status

def _valid_api_keys() -> set[str]:
    raw = os.getenv('SERVICE_API_KEYS', 'test-key-001')
    return {key.strip() for key in raw.split(',') if key.strip()}

async def verify_api_key(x_api_key: str | None = Header(default=None)) -> str:
    if not x_api_key or x_api_key not in _valid_api_keys():
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail='유효한 X-API-Key 헤더가 필요합니다.',
        )
    return x_api_key

### 3.3 schemas.py — 직접 작성

In [ ]:
%%writefile app/schemas.py
from pydantic import BaseModel, Field, field_validator

class PredictRequest(BaseModel):
    text: str = Field(..., min_length=20, max_length=10000, description='요약할 한국어 투자 글')
    max_length: int = Field(default=128, ge=32, le=256)
    min_length: int = Field(default=10, ge=5, le=100)

    @field_validator('text')
    @classmethod
    def text_must_not_be_blank(cls, value: str) -> str:
        cleaned = ' '.join(value.split())
        if len(cleaned) < 20:
            raise ValueError('공백을 제외하고 20자 이상 입력하세요.')
        return cleaned

    @field_validator('min_length')
    @classmethod
    def min_must_be_less_than_max(cls, value: int, info):
        maximum = info.data.get('max_length', 128)
        if value >= maximum:
            raise ValueError('min_length는 max_length보다 작아야 합니다.')
        return value

class PredictResponse(BaseModel):
    success: bool = True
    summary: str
    model: str
    input_characters: int
    truncated: bool
    disclaimer: str = '투자 판단을 위한 참고 자료이며 투자 권유가 아닙니다.'

### 3.4 model_service.py — 직접 작성

In [ ]:
%%writefile app/model_service.py
from functools import lru_cache
from transformers import pipeline

MODEL_ID = 'eenzeenee/t5-small-korean-summarization'
MAX_INPUT_TOKENS = 512

@lru_cache(maxsize=1)
def load_model():
    return pipeline('summarization', model=MODEL_ID, tokenizer=MODEL_ID, device=-1)

def summarize_text(text: str, max_length: int = 128, min_length: int = 10) -> dict:
    model = load_model()
    tokenizer = model.tokenizer
    token_count = len(tokenizer('summarize: ' + text, add_special_tokens=True)['input_ids'])
    output = model(
        'summarize: ' + text,
        max_length=max_length,
        min_length=min_length,
        truncation=True,
        num_beams=4,
        do_sample=False,
        no_repeat_ngram_size=3,
        length_penalty=1.2,
        early_stopping=True,
    )
    summary = output[0]['summary_text'].strip()
    if not summary:
        raise RuntimeError('모델이 빈 요약을 반환했습니다.')
    return {
        'summary': summary,
        'model': MODEL_ID,
        'input_characters': len(text),
        'truncated': token_count > MAX_INPUT_TOKENS,
    }

### 3.5 main.py — 직접 작성

In [ ]:
%%writefile app/main.py
import asyncio
import logging
from functools import partial
from fastapi import Depends, FastAPI, HTTPException, status
from app.auth import verify_api_key
from app.model_service import MODEL_ID, load_model, summarize_text
from app.schemas import PredictRequest, PredictResponse

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
app = FastAPI(title='Investment Article Summarizer', version='1.0.0')

@app.on_event('startup')
async def startup_event():
    loop = asyncio.get_running_loop()
    await loop.run_in_executor(None, load_model)

@app.get('/health')
async def health():
    return {'status': 'ok', 'model': MODEL_ID}

@app.post('/predict', response_model=PredictResponse, dependencies=[Depends(verify_api_key)])
async def predict(request: PredictRequest):
    try:
        loop = asyncio.get_running_loop()
        task = partial(summarize_text, request.text, request.max_length, request.min_length)
        result = await loop.run_in_executor(None, task)
        return PredictResponse(**result)
    except Exception as exc:
        logger.exception('요약 추론 실패')
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail='요약 처리 중 오류가 발생했습니다.',
        ) from exc

### 3.6 frontend/app.py — 직접 작성

In [ ]:
%%writefile frontend/app.py
import os
import requests
import streamlit as st

st.set_page_config(page_title='투자 글 요약', page_icon='📰')
st.title('📰 투자 글 요약')
st.caption('Hugging Face 한국어 T5 모델 기반 · 투자 권유 아님')
api_url = st.sidebar.text_input('API URL', os.getenv('API_URL', 'http://127.0.0.1:8000'))
api_key = st.sidebar.text_input('X-API-Key', os.getenv('SERVICE_API_KEY', 'test-key-001'), type='password')
text = st.text_area('요약할 글', height=300, placeholder='20자 이상의 한국어 투자 관련 글을 붙여 넣으세요.')
st.info('최대 토큰은 상한선입니다. 더 긴 요약이 필요하면 최소 요약 토큰도 높이세요.')
max_length = st.slider('최대 요약 토큰', 64, 256, 160, step=8)
min_length = st.slider('최소 요약 토큰', 10, 100, 40, step=5)
if st.button('요약하기', type='primary'):
    if min_length >= max_length:
        st.warning('최소 요약 토큰은 최대 요약 토큰보다 작아야 합니다.')
    elif len(text.strip()) < 20:
        st.warning('20자 이상 입력하세요.')
    else:
        try:
            with st.spinner('요약 중...'):
                response = requests.post(
                    f'{api_url.rstrip("/")}/predict',
                    headers={'X-API-Key': api_key},
                    json={'text': text, 'max_length': max_length, 'min_length': min_length},
                    timeout=180,
                )
            if response.ok:
                data = response.json()
                st.subheader('요약 결과')
                st.write(data['summary'])
                if data['truncated']:
                    st.warning('입력이 512 토큰을 초과하여 뒷부분이 잘렸습니다.')
                st.caption(data['disclaimer'])
            else:
                st.error(f'API 오류 {response.status_code}: {response.text}')
        except requests.RequestException as exc:
            st.error(f'API 서버에 연결할 수 없습니다: {exc}')

---

## 4. 작업 시간

---

### 4.1 권장 순서



```
Step 1. 모델 선택 + 노트북에서 동작 확인 (섹션 2.3)
        → "이 모델이 내 입력에 대해 결과를 반환하는가?"

Step 2. schemas.py 작성
        → "입력과 출력의 형태를 정의"

Step 3. model_service.py 작성
        → "모델 로드 + 추론 함수"

Step 4. main.py 작성
        → "FastAPI 서버 조립"

Step 5. 서버 실행 + Swagger UI 테스트
        → "API가 동작하는가?"

Step 6. frontend/app.py 작성
        → "Streamlit UI 연결"
```



### 4.2 서버 실행 (Step 5에서 사용)

> ⚠️ **코드를 수정했는데 반영이 안 될 때 — 커널을 재시작하세요.**
>
> `app/main.py`, `app/model_service.py` 등을 고친 뒤 아래 셀을 다시 실행해도,
> 이미 메모리에 올라간 이전 코드가 남아 변경이 반영되지 않을 수 있습니다.
> 코드를 수정했다면 **커널 재시작(Kernel → Restart) 후 맨 위 셀부터 다시 실행**하세요.

In [ ]:
import socket, threading, time, uvicorn
from app.main import app

if 'api_server' not in globals() or not api_thread.is_alive():
    api_config = uvicorn.Config(app, host='127.0.0.1', port=8000, log_level='info')
    api_server = uvicorn.Server(api_config)
    api_server.install_signal_handlers = lambda: None
    api_thread = threading.Thread(target=api_server.run, daemon=True)
    api_thread.start()

def wait_for_port(port: int, timeout: int = 300) -> None:
    deadline = time.time() + timeout
    while time.time() < deadline:
        with socket.socket() as sock:
            if sock.connect_ex(('127.0.0.1', port)) == 0:
                return
        time.sleep(0.5)
    raise TimeoutError(f'{port} 포트가 {timeout}초 안에 열리지 않았습니다.')

wait_for_port(8000)
try:
    from google.colab import output
    print('Colab 셀 안에 Swagger UI를 표시합니다.')
    output.serve_kernel_port_as_iframe(8000, path='/docs', height=800)
except ImportError:
    print('Swagger UI: http://127.0.0.1:8000/docs')

#### Swagger UI 열기

서버가 떴으면 Swagger UI에서 API를 직접 호출해 볼 수 있습니다.  

- 로컬: 브라우저에서 http://localhost:8000/docs
- Colab: localhost 접속이 안 되므로, 아래 셀이 노트북 안에 Swagger UI를 띄웁니다.

> 발표(5.1)의 데모 시연에 이 화면을 그대로 쓸 수 있습니다.


In [ ]:
# Swagger UI를 노트북 안에서 열기
try:
    from google.colab import output as _colab_output   # Colab이면 import에 성공합니다
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import output
    print("✅ Swagger UI를 아래에 띄웁니다 (Colab 프록시 → 포트 8000)")
    output.serve_kernel_port_as_iframe(8000, path="/docs", height="800")
else:
    from IPython.display import IFrame, display
    print("✅ Swagger UI를 아래에 띄웁니다: http://localhost:8000/docs")
    display(IFrame("http://localhost:8000/docs", width="100%", height=800))


### 4.3 API 테스트 템플릿 (Step 5에서 사용)

In [ ]:
import requests
BASE_URL = 'http://127.0.0.1:8000'
valid_payload = {'text': sample_text, 'max_length': 80, 'min_length': 10}

health_response = requests.get(f'{BASE_URL}/health', timeout=10)
unauthorized_response = requests.post(f'{BASE_URL}/predict', json=valid_payload, timeout=10)
invalid_response = requests.post(f'{BASE_URL}/predict', headers={'X-API-Key': 'test-key-001'}, json={'text': '짧음'}, timeout=10)
success_response = requests.post(f'{BASE_URL}/predict', headers={'X-API-Key': 'test-key-001'}, json=valid_payload, timeout=180)

assert health_response.status_code == 200
assert unauthorized_response.status_code == 401
assert invalid_response.status_code == 422
assert success_response.status_code == 200, success_response.text
print('✅ health 200 / 인증 누락 401 / 입력 오류 422 / 정상 추론 200')
print(success_response.json())

### 4.4 프론트엔드 실행 (Step 6에서 사용)

`frontend/app.py`를 작성한 뒤 아래 셀로 띄웁니다.  
노트북 셀에서 `streamlit run`을 직접 실행하면 셀이 끝나지 않으니, 백그라운드로 실행합니다.


In [ ]:
import subprocess, sys, time
if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    streamlit_process.wait(timeout=10)
streamlit_process = subprocess.Popen([
    sys.executable, '-m', 'streamlit', 'run', 'frontend/app.py',
    '--server.port', '8501',
    '--server.address', '0.0.0.0',
    '--server.headless', 'true',
    '--server.enableCORS', 'false',
    '--server.enableXsrfProtection', 'false',
    '--server.enableWebsocketCompression', 'false',
])
wait_for_port(8501, timeout=60)
try:
    from google.colab import output
    print('Colab 셀 안에 Streamlit UI를 표시합니다.')
    output.serve_kernel_port_as_iframe(8501, height=900)
except ImportError:
    print('Streamlit: http://127.0.0.1:8501')

#### 노트북에서 바로 확인

> ⚠️ **화면이 비어 있거나 계속 로딩만 된다면**
> - 위 셀이 `✅`를 출력했는지 먼저 확인하세요.
> - Colab은 런타임에 연결된 상태에서만 iframe을 그립니다. 셀을 다시 실행해보세요.
> - `frontend/app.py`가 비어 있으면 빈 화면이 나옵니다. 3.6을 먼저 완성하세요.


In [ ]:
def show_dashboard(port=8501, height=900):
    """실행 중인 프론트엔드를 노트북 셀 안에 iframe으로 띄웁니다."""
    def port_open(p):
        with contextlib.closing(socket.socket()) as s:
            s.settimeout(0.5)
            return s.connect_ex(("127.0.0.1", p)) == 0

    if not port_open(port):                  # Step 2를 건너뛴 경우
        print(f"⚠️ 포트 {port}에 프론트엔드가 없습니다. Step 2 셀을 먼저 실행하세요.")
        return

    if IN_COLAB:
        from google.colab import output
        print(f"✅ 대시보드를 아래에 띄웁니다 (Colab 프록시 → 포트 {port})")
        output.serve_kernel_port_as_iframe(port, height=str(height))
    else:
        from IPython.display import IFrame, display
        print(f"✅ 대시보드를 아래에 띄웁니다: http://localhost:{port}")
        display(IFrame(f"http://localhost:{port}", width="100%", height=height))

show_dashboard(port=8501, height=900)


### 4.5 막혔을 때 참고할 Day



```
"스키마를 어떻게 정의하지?"        → Day 2 섹션 4, Day 5 섹션 3
"FastAPI 서버 구조가 기억 안 나"   → Day 5 섹션 3, Day 6 섹션 6
"run_in_executor 사용법?"         → Day 3 섹션 4
"인증 적용 방법?"                  → Day 6 섹션 2
"Streamlit에서 API 호출?"         → Day 4 섹션 6, Day 5 섹션 4
"에러 처리?"                      → Day 3 섹션 5
```

---

## 5. 발표 및 회고

---

### 5.1 발표 (개인당 5분)

```
발표 항목:
  1. 어떤 도메인/태스크를 선택했는가?
  2. 어떤 모델을 사용했는가? (선택 이유)
  3. 데모 시연 (Swagger UI 또는 Streamlit)
  4. 구현하면서 가장 어려웠던 부분은?
```

### 5.2 회고

```
스스로 돌아보기:
  - Day 1~7 교안 없이 코드를 작성할 수 있었는가?
  - 어떤 부분에서 교안을 다시 찾아봤는가?
  - 다음에 다시 만든다면 무엇을 다르게 하겠는가?
```

---

## 6. 8일간의 여정 정리

---



### 6.1 Day 1의 문제 → Day 8의 해결

```
Day 1의 문제                           해결한 Day
──────────────────────────────        ──────────
라이브러리가 없음                       Day 1: requirements.txt
모델 구조 코드 필요                     Day 1: model_utils.py 모듈 분리
전처리 로직 누락                       Day 5/7: 전처리 파라미터 저장
비개발자가 사용할 수 없음               Day 4/5/7: Streamlit UI
누구나 API 호출 가능                   Day 6: API Key 인증
스스로 서비스를 만들 수 있는가?          Day 8: 자율 프로젝트 ✅
```



### 6.2 8일간 배운 기술 전체 지도

```
Day 1: 환경 세팅 + 모델 직렬화          "모델을 저장하고 불러온다"
Day 2: FastAPI + Pydantic              "모델을 API로 감싼다"
Day 3: 비동기 처리 + 에러/로깅          "안정적으로 돌아가게 한다"
Day 4: Streamlit + 시스템 아키텍처      "누구나 쓸 수 있게 한다"
Day 5: [프로젝트 1] 정형 데이터 서비스   "따라하며 조립한다"
Day 6: 인증 + 파일 업로드               "보안과 비정형 데이터를 다룬다"
Day 7: [프로젝트 2] 텍스트/이미지 서비스  "패턴을 반복하며 익힌다"
Day 8: [자율 프로젝트] 나만의 서비스      "스스로 만든다"
```



### 6.3 Next Step: MLOps로 가는 길

```
이 과정에서 배운 것:                 다음 과정에서 배울 것:
──────────────────                  ──────────────────
수동으로 서버 실행                    → Docker로 패키징
단일 서버에서 실행                    → 클라우드 배포 (AWS, GCP)
코드 변경 시 수동 재시작              → CI/CD 파이프라인 (자동 빌드/배포)
모델 버전 1개                        → 모델 버전 관리 (MLflow, DVC)
수동 모니터링 (로그 확인)             → 자동 모니터링 (Prometheus, Grafana)
```



> **"코드를 고칠 때마다 매번 서버를 재시작해야 하나요?"**
>
> 그 질문의 답이 MLOps입니다.
> CI/CD 파이프라인이 코드 변경을 감지하면 자동으로 빌드, 테스트, 배포합니다.
> 여러분은 코드를 커밋하기만 하면 됩니다.

---



### ✅ Day 8 최종 체크포인트

**Q1. 본인의 프로젝트에서 Pydantic 검증은 어떤 잘못된 입력을 막아줍니까?**  
빈 문자열과 공백뿐인 문자열, 20자 미만 또는 10,000자 초과 텍스트, 허용 범위를 벗어난 요약 길이, 그리고 `min_length >= max_length`인 요청을 막습니다.

**Q2. Depends(verify_api_key)를 제거하면 어떤 위험이 있습니까?**  
인증되지 않은 사용자가 모델 추론을 무제한 호출해 CPU·메모리와 운영 비용을 소모하고, 서비스 지연이나 장애를 유발할 수 있습니다.

**Q3. run_in_executor를 사용한 이유는 무엇입니까?**  
Transformers 추론은 동기식이며 오래 걸릴 수 있습니다. 별도 스레드에서 실행하여 FastAPI 이벤트 루프가 막히지 않고 다른 요청을 처리할 수 있게 했습니다.

**Q4. Day 1~8 중 가장 많이 참고한 Day는 어디였습니까? 왜?**  
Day 7을 가장 많이 참고했습니다. 텍스트 모델을 서비스 계층으로 분리하고 FastAPI, 비동기 추론, Streamlit 호출까지 연결하는 구조가 이번 요약 서비스와 가장 가까웠기 때문입니다. 인증 부분은 Day 6도 함께 참고했습니다.

**Q5. 이 서비스를 실제로 배포하려면 추가로 무엇이 필요합니까?**  
Docker 이미지, 클라우드 실행 환경과 GPU/CPU 용량 계획, HTTPS와 비밀 키 관리, 요청 제한, 모니터링·로깅, 자동 테스트와 CI/CD, 모델 버전 관리가 필요합니다. 투자 정보 서비스이므로 개인정보·저작권 검토와 투자 권유가 아니라는 고지도 강화해야 합니다.


---

### 📌 Day 8 요약 & 전체 과정 완료

```
오늘 한 일:
  ✅ Day 1~7의 기술을 조합하여 나만의 서비스를 직접 만들었습니다.
  ✅ 교안 없이 설계 → 구현 → 테스트를 경험했습니다.
  ✅ 8일간의 여정을 회고하고, MLOps로 가는 길을 확인했습니다.

8일간의 전체 성과:
  🎉 PyTorch / HuggingFace 모델을 API로 서빙할 수 있습니다.
  🎉 비개발자도 사용 가능한 웹 UI를 붙일 수 있습니다.
  🎉 인증, 에러 처리, 로깅으로 안정적인 서비스를 만들 수 있습니다.
  🎉 스스로 설계하고 구현할 수 있다는 자신감을 얻었습니다.
```

### 제출

06DP08 — 투자 글 요약 서비스


실행 화면 캡처는 제출 범위에서 제외하고, 별도의 `SUBMISSION.md`에 프로젝트 설명, 체크포인트 답변, 회고를 기록합니다.


1. 프로젝트 개요와 실행 방법  
2. 각 세션 및 최종 체크포인트 답변  
3. 프로젝트 회고


수고하셨습니다!